# Flaky Test Prediction Baseline\nMinimal reproducible notebook: synthetic data + Logistic Regression + evaluation metrics.

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import train_test_split

In [ ]:
rng = np.random.default_rng(42)\nn = 500\nX = pd.DataFrame({\n    'fail_rate': rng.uniform(0.0, 0.7, n),\n    'retry_rate': rng.uniform(0.0, 0.5, n),\n    'duration_var': rng.uniform(0.0, 1.0, n),\n    'network_abort_rate': rng.uniform(0.0, 0.3, n),\n    'recent_fail_streak': rng.integers(0, 6, n),\n})\n\nscore = (\n    2.4 * X['fail_rate']\n    + 1.8 * X['retry_rate']\n    + 0.8 * X['duration_var']\n    + 1.6 * X['network_abort_rate']\n    + 0.3 * X['recent_fail_streak']\n    - 1.7\n)\nprob = 1 / (1 + np.exp(-score))\ny = (rng.uniform(0, 1, n) < prob).astype(int)\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.25, random_state=42, stratify=y\n)\nprint('Dataset shape:', X.shape)\nprint('Flaky ratio:', round(float(y.mean()), 3))

In [ ]:
model = LogisticRegression(max_iter=300, class_weight='balanced', random_state=42)\nmodel.fit(X_train, y_train)\npred = model.predict(X_test)\n\nprint('Accuracy:', round(accuracy_score(y_test, pred), 4))\nprint('Classification report:\n', classification_report(y_test, pred, digits=4))\ncm = confusion_matrix(y_test, pred)\nprint('Confusion matrix:')\nprint(cm)